# Optotagging analysis from NWB

This notebook is the NWB-based rewrite of Anna's optotagging pipeline
(`optotagging_analysis.py` + `plotting_funcs.py` + `main.py`). Spikes, waveforms,
peak channels and QC are read from an AIND ephys NWB file loaded with `NWBUtils`.
Laser onsets and stimulation parameters are read from the **raw** Open Ephys asset,
exactly as Anna does.

| Original source                       | Source used here |
|---------------------------------------|----------------------|
| NIDAQ event stream (laser onsets)     | raw NIDAQ events folder (channel 2), same as Anna |
| `*opto.csv` (trial parameters)        | raw `*opto.csv`, same as Anna |
| SpikeInterface `sorting_output`       | `nwb_data.units['spike_times'][u]` |
| waveform extractor templates          | `nwb_data.units['waveform_mean'][u]` |
| `extremum_channels`                   | `nwb_data.units['extremum_channel_index']` |
| probe / stream name                   | `nwb_data.units['device_name']` |
| `default_qc` / `decoder_label`        | `get_units_passed_default_qc(nwb_data)` |

Laser **onset times** are read from the raw Open Ephys NIDAQ digital-input events
(channel 2, label `'2'`) and the **stimulation parameters** from the raw `*opto.csv`.
Only spikes, waveforms, peak channel and QC come from the NWB. The raw
`ecephys_clipped` asset must be attached to the capsule alongside the sorted NWB.


## Setup

In [54]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd

from nwb_utils import NWBUtils
from optotagging_Anna_nwb import (
    OptotaggingAnalysisNWB,
    find_recording_clipped_folder,
    read_opto_trials_csv,
    get_laser_onsets_from_nidaq,
)
import optotagging_Anna_nwb_plotting as opto_plot

print(f"✅ Modules loaded from: {MODULE_PATH}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


## Step 1 — Load the ephys NWB

In [55]:
SESSION_NAME = "ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58"
SAVE_FOLDER = "/root/capsule/scratch/opto_tagging_Anna"

nwb_data = NWBUtils.read_ephys_nwb(session_name=SESSION_NAME)
assert nwb_data is not None, "Failed to load ephys NWB — check the session name."
print("Session:", getattr(nwb_data, "session_id", SESSION_NAME))
print("Probes (device_name):")
from optotagging_Anna_nwb import get_stream_names
print(" ", get_stream_names(nwb_data))


Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58/nwb/ecephys_839480_2026-06-04_13-45-44_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58/nwb/ecephys_839480_2026-06-04_13-45-44_experiment1_recording1.nwb
Session: ecephys_839480_2026-06-04_13-45-44
Probes (device_name):
  ['Probe A', 'Probe B']


## Step 2 — Locate the raw asset (NIDAQ events + opto CSV)

The laser onsets and parameters come from the raw Open Ephys `ecephys_clipped`
folder. This cell finds it, previews the `*opto.csv` parameters, and reads the
NIDAQ channel-2 onsets. If auto-detection fails, set `RECORDING_CLIPPED_FOLDER`
(and optionally `TRIALS_CSV`) explicitly.


In [56]:
# Set these explicitly if auto-detection fails; otherwise leave as None.
RECORDING_CLIPPED_FOLDER = None
TRIALS_CSV = None
LASER_EVENT_ID = "2"   # NIDAQ channel-2 digital-input label
OPTO_RECORDING = 0     # segment index with the laser stimulation
FLIP_NIDAQ = False     # subtract 0.5 s if the sync signal was flipped

clipped = RECORDING_CLIPPED_FOLDER or find_recording_clipped_folder(SESSION_NAME)
print("ecephys_clipped folder:", clipped)

opto_csv = read_opto_trials_csv(clipped, TRIALS_CSV)
print(f"opto CSV rows: {len(opto_csv)}; columns: {list(opto_csv.columns)}")

onsets = get_laser_onsets_from_nidaq(
    clipped, event_id=LASER_EVENT_ID, opto_recording=OPTO_RECORDING, flip_NIDAQ=FLIP_NIDAQ
)
print(f"NIDAQ onsets: {len(onsets)} (should match CSV rows)")
opto_csv.head()


ecephys_clipped folder: /root/capsule/data/ecephys_839480_2026-06-04_13-45-44/ecephys/ecephys_clipped
opto CSV rows: 420; columns: ['site', 'power', 'param_group', 'emission_location', 'duration', 'rise_time', 'num_pulses', 'pulse_interval', 'wavelength', 'type', 'interval']
NIDAQ onsets: 420 (should match CSV rows)


,site,power,param_group,emission_location,duration,rise_time,num_pulses,pulse_interval,wavelength,type,interval
0,0,1.50,train,Probe A,10,1,5,40,473,external_blue,0.89
1,0,0.75,train,Probe A,10,1,5,40,638,external_red,0.85
2,0,1.50,train,Probe A,10,1,5,40,638,external_red,1.04
3,0,0.50,train,Probe A,10,1,5,40,638,external_red,1.08
4,0,0.75,train,Probe A,10,1,5,40,638,external_red,1.12


## Step 3 — Build the analyzer

In [57]:
analysis = OptotaggingAnalysisNWB(
    nwb_data=nwb_data,
    session_name=SESSION_NAME,
    recording_clipped_folder=clipped,
    trials_csv=TRIALS_CSV,
    laser_event_id=LASER_EVENT_ID,
    opto_recording=OPTO_RECORDING,
    flip_NIDAQ=FLIP_NIDAQ,
)

print(f"QC-passing units: {len(analysis.qc_units)}")
print(f"Laser onsets: {len(analysis.laser_onset_times)}")

# trial types present (falls back to a single 'all' type if no 'type' column)
if "type" in analysis.trial_ids.columns:
    trial_types = list(np.unique(analysis.trial_ids["type"]))
else:
    trial_types = ["all"]
print("Trial types:", trial_types)


default_qc flag passed 0 units; reconstructed QC from raw metrics (presence_ratio >= 0.8, isi_violations_ratio <= 0.5).
Number of units passing QC: 385
QC-passing units: 385
Laser onsets: 420
Trial types: ['external_blue', 'external_red']


## Step 4 — Compute laser-response metrics per probe and save CSVs

In [58]:
# Query defines which parameter combinations get analyzed, mirroring main.py.
# Adjust keys to the columns your stimulus table actually has.
powers = list(np.unique(analysis.trial_ids["power"])) if "power" in analysis.trial_ids.columns else [None]
trials_query = {
    "type": trial_types,
    "power": powers,
}
suffixes = [None, "mW"]  # one per trials_query key; matches column naming in main.py

all_metrics = {}
for probe in analysis.get_stream_names():
    metrics = analysis.one_probe_laser_responses(
        trials_query=trials_query,
        probe=probe,
        suffixes=suffixes,
        ignore_onset_offset=True,
        pre_opto_duration=None,  # set to a float (s) to compute pre-stim ISI / rate
    )
    if len(metrics) == 0:
        continue
    metrics = OptotaggingAnalysisNWB.add_best_power_columns(metrics, trial_types)
    all_metrics[probe] = metrics

    Path(SAVE_FOLDER).mkdir(parents=True, exist_ok=True)
    out_csv = Path(SAVE_FOLDER) / f"{analysis.session}_{probe}_laser_response_metrics.csv"
    metrics.to_csv(out_csv, index=False)
    print(f"Saved {out_csv} ({len(metrics)} units)")

/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:754: RuntimeWarning: Mean of empty slice
  float(np.nanmax(lat_valid) - np.nanmin(lat_valid)) if lat_valid.size else np.nan
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:754: RuntimeWarning: Mean of empty slice
  float(np.nanmax(lat_valid) - np.nanmin(lat_valid)) if lat_valid.size else np.nan
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:754: RuntimeWarning: Mean of empty slice
  float(np.nanmax(lat_valid) - np.nanmin(lat_valid)) if lat_valid.size else np.nan
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:754: RuntimeWarning: Mean of empty slice
  float(np.nanmax(lat_valid) - np.nanmin(lat_valid)) if lat_valid.size else np.nan
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:759: RuntimeWarning: Mean of empty slice
  
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:760: RuntimeWarning: Mean of empty slice
  return metrics
/root/

Saved /root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_laser_response_metrics.csv (145 units)
Saved /root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe B_laser_response_metrics.csv (240 units)


## Step 5 — Select tagged units and make plots

Selection criteria mirror `main.py` (adjust the thresholds / trial-type names to
match your data).

In [59]:
def tagged_units(metrics, trial_type, min_sig_pulses=4, max_jitter=0.006, max_isi=0.5):
    """Select tagged units matching Anna's main.py criteria.
    
    - Red: >= 4 sig pulses, jitter < 6ms, ISI ratio < 0.5
    - Blue: == 5 sig pulses, jitter < 6ms, ISI ratio < 0.5
    """
    q = []
    if f"{trial_type}_train_max_num_sig_pulses" in metrics.columns:
        q.append(f"{trial_type}_train_max_num_sig_pulses >= {min_sig_pulses}")
    if f"{trial_type}_train_best_mean_jitter" in metrics.columns:
        q.append(f"{trial_type}_train_best_mean_jitter < {max_jitter}")
    if "pre_stim_isi_ratio" in metrics.columns:
        q.append(f"pre_stim_isi_ratio < {max_isi}")
    if not q:
        return metrics.iloc[0:0]
    return metrics.query(" and ".join(q))


# Process red types first, then blue (excluding red-tagged from blue pool)
red_types = [t for t in trial_types if "red" in t]
blue_types = [t for t in trial_types if "blue" in t]

for probe, metrics in all_metrics.items():
    red_indices = set()

    for trial_type in red_types:
        tagged = tagged_units(metrics, trial_type, min_sig_pulses=4, max_jitter=0.01)
        red_indices.update(tagged.index.tolist())
        unit_ids = tagged["unit_id"].astype(int).tolist()
        print(f"{probe} / {trial_type}: {len(unit_ids)} tagged units -> {unit_ids}")
        if not unit_ids:
            continue
        base = f"{analysis.session}_{probe}_{trial_type}_responsive"
        opto_plot.multi_unit_raster_plot(
            analysis, unit_ids, trial_types, probe, base, save_folder=SAVE_FOLDER
        )
        opto_plot.multi_unit_pulse_plot(
            analysis, unit_ids, metrics, trial_types, probe, base + "_pulse_plot",
            save_folder=SAVE_FOLDER,
        )

    for trial_type in blue_types:
        # Blue requires all 5 pulses significant; exclude red-responsive units
        tagged = tagged_units(metrics, trial_type, min_sig_pulses=5, max_jitter=0.01)
        tagged = tagged[~tagged.index.isin(red_indices)]
        unit_ids = tagged["unit_id"].astype(int).tolist()
        print(f"{probe} / {trial_type}: {len(unit_ids)} tagged units -> {unit_ids}")
        if not unit_ids:
            continue
        base = f"{analysis.session}_{probe}_{trial_type}_responsive"
        opto_plot.multi_unit_raster_plot(
            analysis, unit_ids, trial_types, probe, base, save_folder=SAVE_FOLDER
        )
        opto_plot.multi_unit_pulse_plot(
            analysis, unit_ids, metrics, trial_types, probe, base + "_pulse_plot",
            save_folder=SAVE_FOLDER,
        )

Probe A / external_red: 2 tagged units -> [237, 423]


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:205: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  def _best_power(row: Any, trial_type: str, analysis: Any, trial_type_col: str = "type"):


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_external_red_responsive.png saved


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:297: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_external_red_responsive_pulse_plot.png saved
Probe A / external_blue: 89 tagged units -> [82, 92, 101, 107, 108, 109, 112, 123, 124, 125, 126, 127, 129, 130, 134, 137, 141, 150, 156, 174, 215, 217, 219, 226, 227, 229, 240, 241, 251, 257, 260, 266, 267, 271, 274, 280, 281, 290, 291, 294, 300, 318, 321, 366, 368, 369, 372, 377, 384, 386, 387, 388, 391, 395, 405, 407, 408, 409, 410, 413, 414, 420, 426, 427, 428, 430, 432, 434, 447, 458, 479, 491, 494, 500, 508, 514, 524, 526, 528, 529, 530, 535, 536, 537, 538, 541, 545, 549, 553]


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:205: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  def _best_power(row: Any, trial_type: str, analysis: Any, trial_type_col: str = "type"):


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_external_blue_responsive.png saved


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:297: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.


/root/capsule/scratch/opto_tagging_Anna/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58_Probe A_external_blue_responsive_pulse_plot.png saved
Probe B / external_red: 0 tagged units -> []
Probe B / external_blue: 0 tagged units -> []


## Step 6 — Append opto-tagging results to the NWB units table

This adds an `opto_tagging_Anna` namespace to every unit in the NWB `units` table:

| column | type | meaning |
|--------|------|---------|
| `opto_tagging_Anna_tagged` | bool | passed the tagging criteria |
| `opto_tagging_Anna_tag_type` | str | `external_red` / `external_blue` / `''` |
| `opto_tagging_Anna_criteria` | str | exact query used to tag the unit |
| `opto_tagging_Anna` | str | JSON blob of all laser-response metrics (the CSV row) |

Units that were not analyzed (failed QC or on a non-stimulated probe) get
`tagged=False` and empty values. The columns are added to the in-memory table;
uncomment the export block to also write a standalone NWB copy.


In [ ]:
import optotagging_Anna_nwb_export as opto_export

# Assign tags with the same thresholds used in Step 5
tag_by_probe, criteria_by_type = opto_export.assign_opto_tags(
    all_metrics,
    trial_types,
    red_min_sig_pulses=4,
    blue_min_sig_pulses=5,
    max_jitter=0.01,   # match Step 5
    max_isi=0.5,
)
print("Tagging criteria per type:")
for t, c in criteria_by_type.items():
    print(f"  {t}: {c}")

# Build per-unit columns aligned to the full NWB units table
columns = opto_export.build_opto_tagging_columns(nwb_data, all_metrics, tag_by_probe)

# Append them to the in-memory units table (re-read the NWB first if re-running)
added = opto_export.append_opto_tagging_columns(nwb_data, columns, overwrite=False)
print(f"Added columns to units table: {added}")

# Save the enriched per-unit table (tag + criteria + all metrics) as a CSV
tag_table = opto_export.get_opto_tagging_table(nwb_data, tagged_only=False)
out_csv = Path(SAVE_FOLDER) / f"{analysis.session}_opto_tagging_Anna.csv"
tag_table.to_csv(out_csv, index=False)
n_tagged = int(tag_table["tagged"].sum()) if len(tag_table) else 0
print(f"Saved {out_csv} ({len(tag_table)} analyzed units, {n_tagged} tagged)")

# --- Optional: write a NEW NWB copy with the columns baked in (copies whole file) ---
# source = opto_export.get_nwb_source_path(nwb_data)
# opto_export.export_nwb_with_opto_tagging(
#     source,
#     str(Path(SAVE_FOLDER) / f"{analysis.session}_opto_tagged.nwb"),
#     columns,
#     overwrite=True,
# )


## Step 7 — Select tagged units

Once the `opto_tagging_Anna` columns are on the units table, selecting tagged
units is a one-liner. `select_tagged_units` returns NWB unit indices (optionally
filtered by tag type); `get_opto_tagging_table` returns a tidy DataFrame with the
tag, the criteria used, and every laser-response metric expanded from the JSON blob.


In [ ]:
# Select tagged units easily from the NWB units table
all_tagged = opto_export.select_tagged_units(nwb_data)
print(f"All tagged units ({len(all_tagged)}): {all_tagged.tolist()}")

for ttype in trial_types:
    idx = opto_export.select_tagged_units(nwb_data, tag_type=ttype)
    print(f"{ttype} ({len(idx)}): {idx.tolist()}")

# Full table of just the tagged units, with all metrics expanded
tagged_table = opto_export.get_opto_tagging_table(nwb_data, tagged_only=True)
tagged_table


In [60]:
# Close the NWB IO handle when done
if hasattr(nwb_data, "io"):
    nwb_data.io.close()